In [66]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torch.optim as optim
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import random_split
import numpy as np
import matplotlib as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


Bu model zik zak çiziyor. Batch size, lr kontrol edilecek. Gerekirse de early stopping eklenecek

In [67]:
BLUE = "\033[94m"
GREEN = "\033[92m"
MAGENTA = "\033[95m"
CYAN = "\033[96m"
YELLOW = "\033[93m"
RESET = "\033[0m"

In [68]:
batch_size = 64
lr = 0.001
patience = 5 #early stopping eklersem

In [69]:
cifar_transform = transforms.Compose([transforms.Grayscale(num_output_channels=1),
                                      transforms.Resize((28, 28)),
                                      transforms.ToTensor(),
                                      transforms.Normalize((0.5,), (0.5,))])

transfrom = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5,), (0.5,))])

cifar10 = datasets.CIFAR10(root='./data', train=True, download=True, transform=cifar_transform)
mnist = datasets.MNIST(root='./data', train=True, download=True, transform=transfrom)


In [70]:
class MNISTDataset(Dataset):
    def __init__(self, mnist_data):
        self.mnist_data = mnist_data

    def __len__(self):
        return len(self.mnist_data)

    def __getitem__(self, idx):
        img, label = self.mnist_data[idx]
        return img, label # Flatten the image to a 1D tensor

In [71]:
class CIFAR10Dataset(Dataset):
    def __init__(self, cifar_data):
        self.cifar_data = cifar_data

    def __len__(self):
        return len(self.cifar_data)

    def __getitem__(self, idx):
        img, label = self.cifar_data[idx]
        return img, label #olunca 784 olmayınca 1, 28, 28 !!

In [72]:
cifar10_dataset = CIFAR10Dataset(cifar10)
mnist_dataset = MNISTDataset(mnist)

In [73]:
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

cifar10_len = len(cifar10_dataset)
mnist_len = len(mnist_dataset)
train_len_cifar = int(train_ratio * cifar10_len)
val_len_cifar = int(val_ratio * cifar10_len)

train_len_mnist = int(train_ratio * mnist_len)
val_len_mnist = int(val_ratio * mnist_len)

test_len_cifar = cifar10_len - train_len_cifar - val_len_cifar
test_len_mnist = mnist_len - train_len_mnist - val_len_mnist

dataset_train_cifar, dataset_val_cifar, dataset_test_cifar = random_split(cifar10_dataset, [train_len_cifar, val_len_cifar, test_len_cifar])
dataset_train_mnist, dataset_val_mnist, dataset_test_mnist = random_split(mnist_dataset, [train_len_mnist, val_len_mnist, test_len_mnist])


In [74]:

train_loader_cifar = DataLoader(dataset_train_cifar, batch_size=batch_size, shuffle=True)
val_loader_cifar = DataLoader(dataset_val_cifar, batch_size=batch_size, shuffle=False)
test_loader_cifar = DataLoader(dataset_test_cifar, batch_size=1, shuffle=False)


train_loader_mnist = DataLoader(dataset_train_mnist, batch_size=batch_size, shuffle=True)
val_loader_mnist = DataLoader(dataset_val_mnist, batch_size=batch_size, shuffle=False)
test_loader_mnist = DataLoader(dataset_test_mnist, batch_size=1, shuffle=False)

Burası değişecek!!

In [86]:
class StepFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, treshold=0.0):
        ctx.save_for_backward(input)
        return (input > treshold).float()

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        #grad_input[input <= 0] = 0
        return grad_input, None
    
def step_function(input, treshold=0.0):
    return StepFunction.apply(input, treshold)

#(sigmoid with increasing sharpness over epochs)
def soft_gate(logits, epoch, warmup=3):
    if epoch < warmup:
        sharpness = (epoch + 1) / warmup * 2 #5'ti 
        #return torch.sigmoid(logits * (epoch/warmup))
        return torch.sigmoid(logits * sharpness)
    return step_function(logits)

In [ ]:
def train_model(model, train_loader, val_loader, epochs):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    #buraya entropy eklenenilir! Bak mutlaka

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        train_acc = 0.0
        
        correct = 0
        total = 0

        for inputs, labels in tqdm(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        model.eval()
        valid_loss = 0.0
        valid_acc = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for data, target in tqdm(val_loader):
                data, target = data.to(device), target.to(device)
                output = model(data)# aslında model.forward(data), tahmin üretir

                loss_val = criterion(output, target)#buna neden gerek duyduk ki?
                valid_loss += loss_val.item() * data.size(0)
                _, predicted = torch.max(output.data, 1)
                val_correct += (predicted == target).sum().item()
                val_total += target.size(0)


        train_loss = running_loss / total
        valid_loss = valid_loss / val_total

        train_acc = correct / total
        valid_acc = val_correct / val_total


        print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {valid_loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {valid_acc:.4f}')


In [77]:
class Model1(nn.Module):
    def __init__(self, input_channels, num_classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),

            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(0.5)  # Dropout layer for regularization
        )

        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.encoder(x)
        x = self.classifier(x)
        return x

In [78]:
model1_mnist = Model1(input_channels=1, num_classes=10).to(device)
model1_cifar = Model1(input_channels=1, num_classes=10).to(device)

In [79]:
print(f"{CYAN}CIFAR10 pretraining started...{RESET}")
pretrained_cifar = train_model(model1_cifar, train_loader_cifar, val_loader_cifar, epochs=10)
print(f"{CYAN}CIFAR10 pretraining completed.{RESET}")
print(f"{MAGENTA}MNIST pretraining started...{RESET}")
pretrained_mnist = train_model(model1_mnist, train_loader_mnist, val_loader_mnist, epochs=10)
print(f"{MAGENTA}MNIST pretraining completed.{RESET}")

CIFAR10 pretraining started...


100%|██████████| 118/118 [00:02<00:00, 49.78it/s]


Epoch [1/10], Train Loss: 1.5895, Val Loss: 1.1425, Train Acc: 0.4353, Val Acc: 0.5872


100%|██████████| 118/118 [00:02<00:00, 48.80it/s]


Epoch [2/10], Train Loss: 1.2170, Val Loss: 1.0176, Train Acc: 0.5754, Val Acc: 0.6452


...

Epoch [8/10], Train Loss: 0.0302, Val Loss: 0.0317, Train Acc: 0.9910, Val Acc: 0.9928


100%|██████████| 141/141 [00:02<00:00, 49.43it/s]


Epoch [9/10], Train Loss: 0.0282, Val Loss: 0.0368, Train Acc: 0.9915, Val Acc: 0.9912


100%|██████████| 141/141 [00:02<00:00, 48.53it/s]

Epoch [10/10], Train Loss: 0.0245, Val Loss: 0.0363, Train Acc: 0.9926, Val Acc: 0.9904
MNIST pretraining completed.


New Dataloaders for Combined Datasets

In [80]:
combined_dataset = ConcatDataset([cifar10_dataset, mnist_dataset])


train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

total_len = len(combined_dataset)
train_len = int(train_ratio * total_len)
val_len = int(val_ratio * total_len)
test_len = total_len - train_len - val_len

dataset_train, dataset_val, dataset_test = random_split(combined_dataset, [train_len, val_len, test_len])

In [81]:
class Model2(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=512, encoded_dim=128, num_classes=10):
        super().__init__()
        # Plug in pretrained encoders
        self.encoder1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.encoder2 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1), #sürekli input_channel değiştirmem gerekebilir bunu stabil hale getir!!! 
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.gating_head = nn.Linear(hidden_dim, 2)
        # Gating network (StepGatedModel style)
        self.gate_controller = nn.Sequential(
            nn.Linear(encoded_dim * 2, 256),
            nn.LeakyReLU(0.2),
            nn.LayerNorm(256),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.LayerNorm(128),
            nn.Linear(128, 1)
        )
        
        # Initialize gate controller to near-neutral decisions(weight'ler için, vanishihng gradient sorununu engelliyor)
        for layer in self.gate_controller:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight, gain=0.1)#small bias
                if layer.bias is not None:
                    nn.init.constant_(layer.bias, 0.1)


        #Two separate networks that transform each encoder's features
        self.value_generator1 = nn.Sequential(
            nn.Linear(encoded_dim, encoded_dim),
            nn.ReLU()
        )
        self.value_generator2 = nn.Sequential(
            nn.Linear(encoded_dim, encoded_dim),
            nn.ReLU()
            #StepFunction() #hata verdi
        )
        # Classifier
        self.classifier = nn.Linear(encoded_dim, num_classes)
        #Optional auxiliary network that predicts which dataset the features came from (optional)
        self.dataset_discriminator = nn.Sequential(
            nn.Linear(encoded_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )
        # Add to your model initialization
        self.bn1 = nn.Sequential(
            nn.LayerNorm(encoded_dim),
            nn.Dropout(0.1)
        )  # For MNIST features
        self.bn2 = nn.Sequential(
            nn.LayerNorm(encoded_dim),
            nn.Dropout(0.1)
        )  # For CIFAR10 features


    def forward(self, x1, x2, current_epoch=None):
        # Feature extraction (frozen and pretrained encoders)
        features1 = F.normalize(self.bn1(self.encoder1(x1)), dim=1)#önceden self.encoder1(x1)
        features2 = F.normalize(self.bn2(self.encoder2(x2)), dim=1)

        # Gate decision (hard 0/1)
        combined = torch.cat([features1, features2], dim=1)
        gate_logits = self.gate_controller(combined)
        #gate_decision = step_function(gate_logits)
        if current_epoch is not None:
            gate_decision = soft_gate(gate_logits, current_epoch)
        else:
            gate_decision = step_function(gate_logits)#iyi de o zaman asla step func. kullanılmıyor. burası silinecek sanırım

        # Value generation(adapts features for the classification task)
        values1 = self.value_generator1(features1)
        values2 = self.value_generator2(features2)

        # Strict gating
        #gated_features = torch.where(gate_decision > 0.5, values1, values2)
            # Use straight-through estimator for gradients
        if self.training:
            gate_decision = gate_decision + (step_function(gate_logits) - gate_decision).detach()

        gated_features = gate_decision * values1 + (1 - gate_decision) * values2

        # Outputs
        class_logits = self.classifier(gated_features)
        dataset_logits1 = self.dataset_discriminator(features1)
        dataset_logits2 = self.dataset_discriminator(features2)

        return {
            'class_logits': self.classifier(gated_features),
            'gate_decision': gate_decision,
            'dataset_logits1': self.dataset_discriminator(features1),
            'dataset_logits2': self.dataset_discriminator(features2),
            'raw_features': (features1, features2)
        }

In [ ]:
def train_model2(model, train_loader, val_loader, epochs):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
    criterion = nn.CrossEntropyLoss()#değiştirmeli miyim

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        gate_mnist = 0
        gate_cifar = 0

        for x1, x2, labels in tqdm(train_loader):
            x1, x2 = x1.to(device), x2.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(x1, x2, epoch)

            #burada sıkıntı var sanırım
            loss = criterion(outputs['class_logits'], labels)
            gate_balance_loss = torch.abs(outputs['gate_decision'].mean() - 0.5)

            gate_probs = torch.sigmoid(outputs['gate_decision'])
            entropy_loss = - (gate_probs * torch.log(gate_probs + 1e-10)).mean()
            total_loss = loss + 7.0 * gate_balance_loss + 0.01 * entropy_loss#hyperparameter'larla oynadım

            #total_loss = loss + 0.1 * gate_balance_loss

            total_loss.backward()
            #loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)#silinebilir
            optimizer.step()

            running_loss += loss.item() * x1.size(0)
            _, predicted = outputs['class_logits'].max(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # Count gate decisions
            gate_decisions = outputs['gate_decision'].cpu().detach().numpy()
            #bu kısım sıkıntılı. Enforce ediyoruz sanki modeli(Evet ediyoruz, burası yanlış yeniden bakılacak!!!!!!)
            gate_mnist += (gate_decisions > 0.5).sum()#relu
            gate_cifar += (gate_decisions <= 0.5).sum()#step

        train_loss = running_loss / total
        train_acc = correct / total

        if gate_mnist > gate_cifar:
            print(f"Epoch [{epoch+1}/{epochs}] {BLUE}Gate CIFAR is used{RESET}: {gate_cifar},{MAGENTA}Total:{RESET} {gate_mnist + gate_cifar}")
        else:
            print(f"Epoch [{epoch+1}/{epochs}] {GREEN}Gate MNIST is used{RESET}: {gate_mnist},{MAGENTA}Total:{RESET} {gate_mnist + gate_cifar}")
            
        #print(f"Epoch [{epoch+1}/{epochs}] - Gate MNIST: {gate_mnist}, Gate CIFAR: {gate_cifar}")

        # Validation (optional: add similar gate counting if you want)
        model.eval()
        valid_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for x1, x2, labels in tqdm(val_loader):
                x1, x2 = x1.to(device), x2.to(device)
                labels = labels.to(device)

                outputs = model(x1, x2, epoch)
                
                loss = criterion(outputs['class_logits'], labels)
                valid_loss += loss.item() * x1.size(0)
                _, predicted = outputs['class_logits'].max(1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        valid_loss = valid_loss / val_total
        valid_acc = val_correct / val_total
        scheduler.step()#bunu yanlış yere eklemiş olabilirim

        print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {valid_loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {valid_acc:.4f}')

In [101]:
model2 = Model2().to(device)

'''model2.encoder1.load_state_dict(model1_mnist.encoder.state_dict())
# Copy CIFAR encoder weights to Model2.encoder2
model2.encoder2.load_state_dict(model1_cifar.encoder.state_dict())'''

# 3. Optionally freeze encoders(yapmazsak fine-tuning yapar)
'''for param in model2.encoder1.parameters():
    param.requires_grad = False
for param in model2.encoder2.parameters():
    param.requires_grad = False'''

'for param in model2.encoder1.parameters():\n    param.requires_grad = False\nfor param in model2.encoder2.parameters():\n    param.requires_grad = False'

In [102]:
# Helper: Custom dataset to yield (x1, x2, label)
#bence bu böyle pek mantıklı değil, BAKILACAK!!!!
class PairedDataset2(torch.utils.data.Dataset):
    def __init__(self, mnist_dataset, cifar_dataset):
        self.mnist = mnist_dataset
        self.cifar = cifar_dataset
        self.length = min(len(mnist_dataset), len(cifar_dataset))
        
    def __len__(self):
        return self.length
        
    def __getitem__(self, idx):
        mnist_img, mnist_label = self.mnist[idx]
        cifar_img, cifar_label = self.cifar[idx]
        # Use MNIST label as ground truth (assuming aligned classes)
        return mnist_img, cifar_img, mnist_label
    
# Make sure your splits are of equal length
min_train_len = min(len(dataset_train_mnist), len(dataset_train_cifar))
dataset_train_mnist = torch.utils.data.Subset(dataset_train_mnist, range(min_train_len))
dataset_train_cifar = torch.utils.data.Subset(dataset_train_cifar, range(min_train_len))

# Similarly for validation and test
min_val_len = min(len(dataset_val_mnist), len(dataset_val_cifar))
dataset_val_mnist = torch.utils.data.Subset(dataset_val_mnist, range(min_val_len))
dataset_val_cifar = torch.utils.data.Subset(dataset_val_cifar, range(min_val_len))

min_test_len = min(len(dataset_test_mnist), len(dataset_test_cifar))
dataset_test_mnist = torch.utils.data.Subset(dataset_test_mnist, range(min_test_len))
dataset_test_cifar = torch.utils.data.Subset(dataset_test_cifar, range(min_test_len))

paired_train_loader = DataLoader(PairedDataset2(dataset_train_mnist, dataset_train_cifar), batch_size=batch_size, shuffle=True)
paired_val_loader = DataLoader(PairedDataset2(dataset_val_mnist, dataset_val_cifar), batch_size=batch_size, shuffle=False)
paired_test_loader = DataLoader(PairedDataset2(dataset_test_mnist, dataset_test_cifar), batch_size=1, shuffle=False)

print("Paired training started...")
train_model2(model2, paired_train_loader, paired_val_loader, epochs=10)
print("Paired training completed.")

#normalde ilk runladığımda iki datasetten de gate için kullanıyordu ama sonra sadece mnsit için kullanmaya başladı(bakılacak!!!!)

Paired training started...


100%|██████████| 547/547 [00:17<00:00, 31.41it/s]


Epoch [1/10] Gate MNIST is used: 17384,Total: 35000


100%|██████████| 118/118 [00:02<00:00, 56.82it/s]


Epoch [1/10], Train Loss: 1.6876, Val Loss: 0.3141, Train Acc: 0.4091, Val Acc: 0.9627


...

Epoch [9/10], Train Loss: 1.1078, Val Loss: 2.2742, Train Acc: 0.5759, Val Acc: 0.1244


100%|██████████| 547/547 [00:18<00:00, 29.38it/s]


Epoch [10/10] Gate CIFAR is used: 17209,Total: 35000


100%|██████████| 118/118 [00:02<00:00, 55.40it/s]

Epoch [10/10], Train Loss: 1.1636, Val Loss: 2.1402, Train Acc: 0.5538, Val Acc: 0.1819
Paired training completed.
